# 02 — RAIS: Análise Longitudinal 2023–2025
## TCC MBA BI&A USP — CBO 2112-20 Cientista de Dados
**Tema:** potenciais impactos da supressão de células na análise da equidade de gênero.

### Roteiro (executar de cima para baixo, "Executar tudo")
| Seção | Conteúdo | Alimenta no artigo |
|---|---|---|
| **1** | Configuração, parâmetros e funções auxiliares | Material e Métodos |
| **2** | Validação das bases contra o Sumário Executivo da RAIS | Material e Métodos / Resultados 4.1 |
| **3** | Resultados descritivos da ocupação (Tabelas 1 a 3 e testes) | Resultados 4.2 e 4.3 |
| **4** | Simulação de supressão (Tabelas 4 a 8 e cruzamentos finos) | Resultados 4.4 a 4.6 |
| **5** | Análises complementares (salário, raça/cor, escolaridade, porte, vínculo, completude) | Apêndice |
| **6** | Resumo executivo e exportação das tabelas | — |

### Pré-requisitos
- Bancos `rais_2023.duckdb`, `rais_2024.duckdb` e `rais_2025.duckdb` (um por pasta de ano), gerados pelo notebook `01_carga_rais.ipynb`, em `DADOS_PATH` (variável de ambiente `RAIS_DADOS_PATH`; padrão: pasta do Google Drive).
- `rais_dicionarios.py` (dicionários de códigos da RAIS) na raiz do repositório.

### Definições fixas do estudo
- **Estoque:** vínculo ativo em 31/12 (`Ind Vínculo Ativo 31/12 = 1`), excluídos os vínculos abandonados (`Ind Vínculo Abandonado = 0`), com sexo identificado (códigos 1 e 2). Definido **uma única vez** (seção 1) e reutilizado em todas as consultas.
- **Célula analítica:** combinação de categorias (sexo × UF, sexo × região, etc.), com a grade **completa** (células sem vínculo contam como zero e **não** são suprimidas).
- **Supressão primária:** oculta células com 1 a k−1 vínculos, para k = 3, 5 e 10. **Complementar (heurística):** oculta células adicionais que permitiriam deduzir a oculta por diferença.

### Contexto normativo
- Desde o ano-base 2023, a RAIS é captada exclusivamente pelo eSocial (Portaria Consolidada MTE nº 1/2025).
- A CBO 2112-20 foi incluída na classificação em 20/03/2023; 2023 é ano parcial para a ocupação.
- Lei nº 14.611/2023 (Igualdade Salarial): referência nacional do Relatório de Transparência Salarial (painel dez/2025, estabelecimentos com 100+ empregados): razão M/H da remuneração média de 78,7% (gap de 21,3%).

# 1. Configuração

In [ ]:
# 1.1 Ambiente (Colab) e dependências
import sys, subprocess

EM_COLAB = 'google.colab' in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])
    import duckdb

In [ ]:
# 1.2 Bibliotecas e estilo gráfico
import os, shutil, warnings
from math import sqrt, erfc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'figure.dpi': 150, 'axes.grid': True, 'grid.alpha': 0.3,
})

In [ ]:
# 1.3 Caminhos e parâmetros da análise
DADOS_PATH = os.environ.get('RAIS_DADOS_PATH', '/content/drive/MyDrive/MBA_USP_BIA/TCC/CAGED')
DB_PATH    = lambda ano: f'{DADOS_PATH}/{ano}/rais_{ano}.duckdb'
FIG_DIR    = os.path.join(DADOS_PATH, 'figuras'); os.makedirs(FIG_DIR, exist_ok=True)
TAB_DIR    = os.path.join(DADOS_PATH, 'tabelas'); os.makedirs(TAB_DIR, exist_ok=True)

CBO  = '211220'
ANOS = [2023, 2024, 2025]
KS   = (3, 5, 10)                     # limiares de supressão


# Definição única do estoque (usada por todas as consultas via {ESTOQUE})
FILTRO_ESTOQUE = ('"Ind Vínculo Ativo 31/12 - Código" = \'1\' '
                  'AND "Ind Vínculo Abandonado - Código" = \'0\'')

# Totais oficiais de vínculos ativos (Sumário Executivo RAIS 2025, Tabela 8 — série revisada;
# o valor de 2023 também consta do Sumário 2024. O Sumário 2023 original traz 54.706.385.)
REF_SUMARIO = {
    2023: dict(Homens=30_615_645, Mulheres=24_700_969, Total=55_316_614),
    2024: dict(Homens=31_371_912, Mulheres=25_760_244, Total=57_132_156),
    2025: dict(Homens=32_562_771, Mulheres=27_408_174, Total=59_970_945),
}

# Geografia
UFS = ['AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
       'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO']
REGIOES = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
SEXOS = ['Masculino', 'Feminino']
ROTULO_SEXO = {'Masculino': 'Homens', 'Feminino': 'Mulheres'}

In [ ]:
# 1.4 Dicionários de códigos da RAIS (arquivo rais_dicionarios.py)
# Procura na raiz do repositório, na pasta do notebook e em DADOS_PATH
import tempfile
_candidatos = [os.path.join(os.getcwd(), 'rais_dicionarios.py'),
               os.path.join(os.getcwd(), '..', 'rais_dicionarios.py'),
               os.path.join(DADOS_PATH, 'rais_dicionarios.py')]
_dic_origem = next((c for c in _candidatos if os.path.exists(c)), None)
assert _dic_origem, 'rais_dicionarios.py não encontrado. Procurado em: ' + ', '.join(_candidatos)
_tmp = tempfile.gettempdir()
shutil.copy(_dic_origem, os.path.join(_tmp, 'rais_dicionarios.py'))
sys.path.insert(0, _tmp)

from rais_dicionarios import (
    SEXO, ESCOLARIDADE, ESCOLARIDADE_GRUPO, FAIXA_ETARIA, FAIXA_ETARIA_ORDEM,
    RACA_COR, RACA_COR_GRUPO, TAMANHO_ESTAB, TAMANHO_ESTAB_PORTE, TAMANHO_PORTE_ORDEM,
    FAIXA_REMUN_MEDIA, FAIXA_REMUN_ORDEM, CORES_GENERO, CORES_RACA,
    CORES_ESCOLARIDADE, TIPO_VINCULO_GRUPO, decodificar
)
print('✅ Dicionários carregados de', _dic_origem)

In [ ]:
# 1.5 Funções auxiliares
def query_anos(sql_template, anos=ANOS):
    """Executa o SQL em cada banco anual (somente leitura) e concatena, com coluna 'ano'.
    Placeholders: {CBO}, {ano} e {ESTOQUE} (filtro do estoque)."""
    frames = []
    for ano in anos:
        con = duckdb.connect(DB_PATH(ano), read_only=True)
        try:
            df = con.execute(sql_template.format(ano=ano, CBO=CBO, ESTOQUE=FILTRO_ESTOQUE)).df()
        finally:
            con.close()
        if 'ano' not in df.columns:
            df.insert(0, 'ano', ano)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

def br(x, casas=0):
    """Formato brasileiro: milhar com ponto, decimal com vírgula."""
    return f'{x:,.{casas}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def inteiros(df, cols):
    df[cols] = df[cols].astype('int64')
    return df

def salvar_fig(nome):
    caminho = os.path.join(FIG_DIR, nome)
    plt.savefig(caminho, bbox_inches='tight')
    print(f'💾 {caminho}')

TABELAS = {}   # tabelas finais, exportadas na seção 6
print('✅ Funções auxiliares prontas.')

# 2. Validação das bases
Antes de qualquer recorte ocupacional, confere-se (i) a integridade de cada arquivo anual e (ii) se a contagem por sexo reproduz os totais oficiais do Sumário Executivo.

> **Nota sobre a série de 2023:** o total de 2023 nos microdados (55.316.614) coincide com a série revisada divulgada nos Sumários de 2024 e 2025, e difere do Sumário 2023 original (54.706.385), que não foi retificado formalmente.

In [ ]:
# 2.1 Integridade dos arquivos: tamanho, data, registros brutos e colunas obrigatórias
COLUNAS_OBRIGATORIAS = [
    'Sexo - Código', 'Ind Vínculo Ativo 31/12 - Código', 'Ind Vínculo Abandonado - Código',
    'CBO 2002 Ocupação - Código', 'uf', 'regiao_geografica',
    'Escolaridade Após 2005 - Código', 'Raça Cor - Código',
    'Tamanho Estabelecimento - Código', 'Tipo Vínculo - Código',
    'Vl Rem Média Nom', 'Município - Código',
]
linhas = []
for ano in ANOS:
    caminho = DB_PATH(ano)
    st = os.stat(caminho)
    con = duckdb.connect(caminho, read_only=True)
    try:
        cols = set(con.execute('DESCRIBE rais_vinc_brasil').df()['column_name'])
        n_bruto = con.execute('SELECT count(*) FROM rais_vinc_brasil').fetchone()[0]
    finally:
        con.close()
    linhas.append({'Ano': ano, 'Arquivo': os.path.basename(caminho),
                   'Tamanho (GB)': round(st.st_size / 1e9, 3),
                   'Modificado em': pd.Timestamp(st.st_mtime, unit='s').strftime('%Y-%m-%d %H:%M'),
                   'Registros brutos': br(n_bruto), 'Colunas': len(cols),
                   'Colunas ausentes': sorted(set(COLUNAS_OBRIGATORIAS) - cols) or '—'})
display(pd.DataFrame(linhas))
print('Atenção: tamanhos/contagens idênticos entre anos indicariam carga duplicada.')

In [ ]:
# 2.2 Estoque total por sexo × Sumário Executivo
sql_total = """
SELECT COUNT(*) AS Total,
       SUM(CASE WHEN "Sexo - Código" = '1' THEN 1 ELSE 0 END) AS Homens,
       SUM(CASE WHEN "Sexo - Código" = '2' THEN 1 ELSE 0 END) AS Mulheres
FROM rais_vinc_brasil
WHERE {ESTOQUE} AND "Sexo - Código" IN ('1','2')
"""
df_total = query_anos(sql_total).rename(columns={'ano': 'Ano'})
df_total = inteiros(df_total, ['Total', 'Homens', 'Mulheres'])
df_total['% Mulheres'] = df_total['Mulheres'] / df_total['Total'] * 100

val = df_total.copy()
val['Total (Sumário)']   = val['Ano'].map(lambda a: REF_SUMARIO[a]['Total'])
val['Diferença total']   = val['Total'] - val['Total (Sumário)']
val['Dif. homens']       = val['Homens'] - val['Ano'].map(lambda a: REF_SUMARIO[a]['Homens'])
val['Dif. mulheres']     = val['Mulheres'] - val['Ano'].map(lambda a: REF_SUMARIO[a]['Mulheres'])
val['Status'] = np.where((val[['Diferença total', 'Dif. homens', 'Dif. mulheres']] == 0).all(axis=1), '✅ confere', '❌ difere')

print('Validação — vínculos ativos em 31/12 (excluídos abandonados e sem sexo):')
display(val.style.format({'Total': br, 'Homens': br, 'Mulheres': br, 'Total (Sumário)': br,
                          'Diferença total': br, 'Dif. homens': br, 'Dif. mulheres': br,
                          '% Mulheres': lambda v: br(v, 2) + '%'}).hide(axis='index'))
if (val['Status'] != '✅ confere').any():
    print('⚠️ Há divergência com o Sumário: investigar antes de prosseguir.')
else:
    print('✅ Os três anos reproduzem integralmente a série oficial.')
TABELAS['T1_validacao_total'] = val

# 3. Resultados descritivos — CBO 2112-20
## 3.1 Evolução dos vínculos e participação feminina
> **Nota metodológica:** a CBO 2112-20 foi incluída em 20/03/2023; 2023 é ano parcial e o crescimento do estoque reflete sobretudo a adoção do código.

In [ ]:
# 3.1.1 Estoque da ocupação por sexo e ano
sql_cbo = """
SELECT COUNT(*) AS Total,
       SUM(CASE WHEN "Sexo - Código" = '1' THEN 1 ELSE 0 END) AS Masculino,
       SUM(CASE WHEN "Sexo - Código" = '2' THEN 1 ELSE 0 END) AS Feminino
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2')
"""
df_cbo = query_anos(sql_cbo).rename(columns={'ano': 'Ano'})
df_cbo = inteiros(df_cbo, ['Total', 'Masculino', 'Feminino'])
df_cbo['% Feminino']  = df_cbo['Feminino']  / df_cbo['Total'] * 100
df_cbo['% Masculino'] = df_cbo['Masculino'] / df_cbo['Total'] * 100
df_cbo['Crescimento (%)'] = df_cbo['Total'].pct_change() * 100

tab2 = df_cbo[['Ano', 'Total', 'Masculino', 'Feminino', '% Feminino']]
print('Tabela 2 — Vínculos ativos, CBO 2112-20, por sexo:')
display(tab2.style.format({'Total': br, 'Masculino': br, 'Feminino': br,
                           '% Feminino': lambda v: br(v, 1) + '%'}).hide(axis='index'))
TABELAS['T2_cbo_por_sexo'] = tab2

In [ ]:
# 3.1.2 Comparativo: ocupação × mercado formal total
comp = df_total[['Ano', 'Total', 'Mulheres', '% Mulheres']].merge(
    df_cbo[['Ano', 'Total', 'Feminino', '% Feminino']], on='Ano', suffixes=(' (mercado)', ' (CBO)'))
comp = comp.rename(columns={'Total (mercado)': 'Total mercado formal', 'Mulheres': 'Mulheres no mercado formal',
                            '% Mulheres': '% Mulheres no mercado formal', 'Total (CBO)': 'Total Cientistas de Dados',
                            'Feminino': 'Mulheres Cientistas de Dados', '% Feminino': '% Mulheres Cientistas de Dados'})
comp['Diferença (p.p.)'] = comp['% Mulheres no mercado formal'] - comp['% Mulheres Cientistas de Dados']
print('Tabela 1 (complemento) — participação feminina: mercado formal × ocupação')
display(comp.style.format({c: br for c in comp.columns if comp[c].dtype.kind == 'i'}
                          | {c: (lambda v: br(v, 2)) for c in comp.columns if comp[c].dtype.kind == 'f'}).hide(axis='index'))
TABELAS['T1_comparativo'] = comp

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
# (a) Total de vínculos
axes[0].bar(df_cbo['Ano'], df_cbo['Total'], color='#455A64', width=0.5)
axes[0].set_title('Total de Vínculos\nCientista de Dados (CBO 2112-20)')
axes[0].set_xlabel('Ano'); axes[0].set_ylabel('Nº de Vínculos'); axes[0].set_xticks(ANOS)
for _, r in df_cbo.iterrows():
    axes[0].text(r['Ano'], r['Total'] + 20, br(r['Total']), ha='center', fontsize=10, fontweight='bold')
    if not pd.isna(r['Crescimento (%)']):
        axes[0].text(r['Ano'], r['Total'] / 2, f"+{r['Crescimento (%)']:.0f}%", ha='center', color='white', fontsize=9)
# (b) Participação por sexo
axes[1].bar(df_cbo['Ano'], df_cbo['% Masculino'], label='Masculino', color=CORES_GENERO['Masculino'], width=0.5)
axes[1].bar(df_cbo['Ano'], df_cbo['% Feminino'], bottom=df_cbo['% Masculino'], label='Feminino',
            color=CORES_GENERO['Feminino'], width=0.5)
axes[1].axhline(50, color='black', linestyle='--', lw=0.8, alpha=0.5, label='Paridade')
axes[1].set_title('Participação por Sexo (%)\nCientista de Dados (CBO 2112-20)')
axes[1].set_xlabel('Ano'); axes[1].set_ylabel('%'); axes[1].set_ylim(0, 100); axes[1].set_xticks(ANOS); axes[1].legend()
for _, r in df_cbo.iterrows():
    axes[1].text(r['Ano'], r['% Masculino'] / 2, f"{r['% Masculino']:.1f}%", ha='center', color='white', fontweight='bold')
    axes[1].text(r['Ano'], r['% Masculino'] + r['% Feminino'] / 2, f"{r['% Feminino']:.1f}%", ha='center', color='white', fontweight='bold')
# (c) Ocupação × mercado formal
axes[2].plot(df_total['Ano'], df_total['% Mulheres'], '--s', color='#9E9E9E', lw=2.5, markersize=8, label='Mercado formal total')
axes[2].plot(df_cbo['Ano'], df_cbo['% Feminino'], '-o', color=CORES_GENERO['Feminino'], lw=2.5, markersize=8, label='Cientista de Dados')
axes[2].axhline(50, color='black', linestyle=':', lw=0.8, alpha=0.4, label='Paridade (50%)')
axes[2].set_title('Participação Feminina (%)\nOcupação × Mercado Formal')
axes[2].set_xlabel('Ano'); axes[2].set_ylabel('%'); axes[2].set_ylim(0, 60); axes[2].set_xticks(ANOS); axes[2].legend()
plt.tight_layout()
salvar_fig('fig1_evolucao_participacao.png')
plt.show()

In [ ]:
# 3.1.3 Estabilidade da participação feminina (teste z de duas proporções e qui-quadrado)
def z_duas_proporcoes(x1, n1, x2, n2):
    p1, p2, p = x1 / n1, x2 / n2, (x1 + x2) / (n1 + n2)
    z = (p2 - p1) / sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    return z, erfc(abs(z) / sqrt(2))

d = df_cbo.set_index('Ano')
testes = []
for a1, a2 in [(2023, 2024), (2023, 2025), (2024, 2025)]:
    z, p = z_duas_proporcoes(d.loc[a1, 'Feminino'], d.loc[a1, 'Total'], d.loc[a2, 'Feminino'], d.loc[a2, 'Total'])
    testes.append({'Comparação': f'{a1} × {a2}', 'z': round(z, 3), 'p-valor': round(p, 3)})
chi2, p_chi, gl = chi2_contingency(d[['Masculino', 'Feminino']].values)[:3]
testes.append({'Comparação': 'Global (3 anos × 2 sexos)', 'z': f'χ²={chi2:.3f} (gl={gl})', 'p-valor': round(p_chi, 3)})
tab_testes = pd.DataFrame(testes)
display(tab_testes)
TABELAS['T2b_testes_estabilidade'] = tab_testes

## 3.2 Distribuição por UF (Tabela 3)
> Células com poucos vínculos são exibidas **sem supressão**, para demonstração. Os dados são públicos e sem identificação direta. Em "–", a UF não tem vínculos na ocupação (zero estrutural, não supressão).

In [ ]:
# 3.2.1 Contagens por UF, sexo e ano (grade completa de 27 UFs)
sql_uf = """
SELECT uf AS UF, COUNT(*) AS Total,
       SUM(CASE WHEN "Sexo - Código" = '1' THEN 1 ELSE 0 END) AS Homens,
       SUM(CASE WHEN "Sexo - Código" = '2' THEN 1 ELSE 0 END) AS Mulheres
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2')
GROUP BY uf
"""
df_uf_obs = query_anos(sql_uf).rename(columns={'ano': 'Ano'})
grade_uf = pd.MultiIndex.from_product([UFS, ANOS], names=['UF', 'Ano'])
df_uf = (df_uf_obs.set_index(['UF', 'Ano'])[['Homens', 'Mulheres', 'Total']]
         .reindex(grade_uf, fill_value=0).astype('int64').reset_index())
# Conferência: soma das UFs = total da ocupação
conf = df_uf.groupby('Ano')[['Homens', 'Mulheres', 'Total']].sum()
assert (conf['Total'].values == df_cbo['Total'].values).all(), 'Soma das UFs difere do total da ocupação'
print('✅ Soma das 27 UFs = total da ocupação nos três anos.')
fora = sorted(set(df_uf_obs['UF']) - set(UFS))
if fora: print('⚠️ UFs fora da lista oficial:', fora)

In [ ]:
# 3.2.2 Tabela 3 pivotada (H/M/T e %M) com linha de total
brasil = df_uf.groupby('Ano')[['Homens', 'Mulheres', 'Total']].sum().reset_index().assign(UF='TOTAL BRASIL')
t3 = pd.concat([df_uf, brasil], ignore_index=True)
t3['%M'] = (t3['Mulheres'] / t3['Total'] * 100).where(t3['Total'] > 0, 0)
t3['Valor'] = t3.apply(lambda r: '–' if r['Total'] == 0 else
                       f"{br(r['Homens'])}/{br(r['Mulheres'])}/{br(r['Total'])} ({br(r['%M'], 1)}%)", axis=1)
tabela3 = t3.pivot(index='UF', columns='Ano', values='Valor')[ANOS].reindex(UFS + ['TOTAL BRASIL'])
print('Tabela 3 — vínculos de homens/mulheres/total (e % de mulheres) por UF:')
display(tabela3)
TABELAS['T3_vinculos_por_uf'] = tabela3.reset_index()

In [ ]:
# 3.2.3 Indicadores de concentração e de zeros (citados no texto)
conc = []
for ano in ANOS:
    d_ = df_uf[df_uf['Ano'] == ano].set_index('UF')
    sp = d_.loc['SP']
    conc.append({'Ano': ano,
                 'SP: % das mulheres': 100 * sp['Mulheres'] / d_['Mulheres'].sum(),
                 'SP: % dos homens':   100 * sp['Homens'] / d_['Homens'].sum(),
                 'UFs com vínculos': int((d_['Total'] > 0).sum()),
                 'UFs sem mulheres (com homens)': ', '.join(d_.index[(d_['Mulheres'] == 0) & (d_['Homens'] > 0)]) or '—'})
display(pd.DataFrame(conc).round(1))

# 4. Simulação de supressão de células
Todos os cruzamentos usam a **mesma definição de célula e de métricas**, para serem comparáveis:
- grade completa (zeros incluídos); **suprimida** = célula com 1 a k−1 vínculos;
- métricas: % de células suprimidas, % de vínculos ocultados (por sexo), participação feminina visível e UFs/regiões em que a razão mulheres/homens ainda é calculável.

In [ ]:
# 4.1 Funções de simulação
def grade_completa(df, dims, universos):
    """df com colunas ['ano','sexo',*dims,'n'] -> grade completa (zeros incluídos)."""
    idx = pd.MultiIndex.from_product([ANOS, SEXOS] + [universos[d] for d in dims], names=['ano', 'sexo'] + dims)
    return df.groupby(['ano', 'sexo'] + dims)['n'].sum().reindex(idx, fill_value=0).reset_index()

def supressao_primaria(grade, ks=KS):
    """Supressão primária: oculta células com 0 < n < k."""
    out = []
    for k in ks:
        g = grade.assign(suprimida=(grade['n'] > 0) & (grade['n'] < k))
        for (ano, sexo), s in g.groupby(['ano', 'sexo']):
            tot = s['n'].sum()
            ocultos = s.loc[s['suprimida'], 'n'].sum()
            out.append({'k': k, 'Ano': ano, 'Sexo': sexo, 'celulas': len(s),
                        'celulas_com_vinculo': int((s['n'] > 0).sum()),
                        'celulas_suprimidas': int(s['suprimida'].sum()),
                        '% celulas suprimidas': 100 * s['suprimida'].mean(),
                        'vinculos_ocultados': int(ocultos),
                        '% vinculos ocultados': 100 * ocultos / tot if tot > 0 else 0.0})
    return pd.DataFrame(out)

def simular_complementar(df_un, ano, k, col, universo):
    """Primária + complementar (heurística gulosa) para um cruzamento sexo × unidade.
    df_un: colunas ['Ano', col, 'Homens', 'Mulheres']."""
    d = (df_un[df_un['Ano'] == ano].set_index(col)[['Homens', 'Mulheres']]
         .reindex(universo).fillna(0).astype(int))
    H, M = d['Homens'].values, d['Mulheres'].values
    prim = ((H > 0) & (H < k), (M > 0) & (M < k))
    sH, sM = prim[0].copy(), prim[1].copy()
    mudou = True
    while mudou:
        mudou = False
        for i in range(len(universo)):        # total da unidade publicado -> par de células
            if H[i] + M[i] > 0 and sH[i] != sM[i]:
                sH[i] = sM[i] = True; mudou = True
        for s, v in ((sH, H), (sM, M)):       # total nacional publicado -> coluna
            if s.sum() == 1:
                cand = [(v[i], i) for i in range(len(universo)) if not s[i] and v[i] > 0]
                if cand:
                    s[min(cand)[1]] = True; mudou = True
    def met(a, b):
        base = ((H > 0) & (M > 0)).sum()
        ok = ((~a) & (~b) & (H > 0) & (M > 0)).sum()
        return dict(cel_H=int(a.sum()), cel_M=int(b.sum()),
                    oculto_H=100 * H[a].sum() / H.sum(), oculto_M=100 * M[b].sum() / M.sum(),
                    part_visivel=100 * M[~b].sum() / (H[~a].sum() + M[~b].sum()),
                    razao_calculavel=int(ok), razao_base=int(base))
    return 100 * M.sum() / (H.sum() + M.sum()), met(*prim), met(sH, sM)

def rodar_simulacao(df_un, col, universo):
    linhas = []
    for ano in ANOS:
        for k in KS:
            orig, p, c = simular_complementar(df_un, ano, k, col, universo)
            linhas.append({'Ano': ano, 'k': k, 'Original (%)': orig,
                           **{f'prim_{a}': b for a, b in p.items()},
                           **{f'comp_{a}': b for a, b in c.items()}})
    return pd.DataFrame(linhas)
print('✅ Funções de simulação prontas.')

## 4.2 Sexo × UF (cruzamento principal) — Tabelas 4 e 5

In [ ]:
# 4.2.1 Tabela 4 — supressão primária, sexo × UF (27 células por sexo e ano)
long_uf = (df_uf.melt(id_vars=['UF', 'Ano'], value_vars=['Homens', 'Mulheres'], var_name='sexo', value_name='n')
           .assign(sexo=lambda x: x['sexo'].map({'Homens': 'Masculino', 'Mulheres': 'Feminino'}))
           .rename(columns={'Ano': 'ano'}))
grade_a = grade_completa(long_uf, ['UF'], {'UF': UFS})
res_uf = supressao_primaria(grade_a)

tab4 = res_uf.assign(Sexo=res_uf['Sexo'].map(ROTULO_SEXO)).sort_values(['k', 'Ano', 'Sexo'], ascending=[True, True, True])
tab4 = tab4[['k', 'Ano', 'Sexo', 'celulas', 'celulas_suprimidas', '% celulas suprimidas', 'vinculos_ocultados', '% vinculos ocultados']]
print('Tabela 4 — análise de supressão de células (sexo × UF):')
display(tab4.style.format({'% celulas suprimidas': lambda v: br(v, 1), '% vinculos ocultados': lambda v: br(v, 1)}).hide(axis='index'))
TABELAS['T4_supressao_primaria'] = tab4

In [ ]:
# 4.2.2 Tabela 5 — % de vínculos ocultados por sexo e razão mulheres/homens
p = res_uf.pivot_table(index=['k', 'Ano'], columns='Sexo', values='% vinculos ocultados').reset_index()
tab5 = p.rename(columns={'Masculino': 'Homens (%)', 'Feminino': 'Mulheres (%)'})
tab5['Razão M/H'] = tab5['Mulheres (%)'] / tab5['Homens (%)']
print('Tabela 5 — % de vínculos ocultados por sexo (razão = % mulheres ÷ % homens):')
display(tab5.style.format({'Homens (%)': lambda v: br(v, 1) + '%', 'Mulheres (%)': lambda v: br(v, 1) + '%',
                           'Razão M/H': lambda v: br(v, 1)}).hide(axis='index'))
TABELAS['T5_razao_ocultados'] = tab5

## 4.3 Viés, comparabilidade e supressão complementar (sexo × UF) — Tabelas 6 a 8

In [ ]:
# 4.3.1 Simulação completa (primária + complementar)
sim_uf = rodar_simulacao(df_uf, 'UF', UFS)

# Tabela 6 — participação feminina nacional recalculada (supressão primária)
t6 = sim_uf.pivot(index='Ano', columns='k', values='prim_part_visivel')
t6.insert(0, 'Original', sim_uf.groupby('Ano')['Original (%)'].first())
t6.columns = ['Original'] + [f'k = {k}' for k in KS]
print('Tabela 6 — participação feminina (%) recalculada com as células visíveis (primária):')
display(t6.round(2))
print('Variação em pontos percentuais:')
display((t6[[f'k = {k}' for k in KS]].sub(t6['Original'], axis=0)).round(2))

# Tabela 7 — UFs com razão M/H calculável
t7 = sim_uf.assign(**{'Razão M/H calculável': lambda x: x['prim_razao_calculavel'].astype(str) + ' de ' + x['prim_razao_base'].astype(str)}) \
           .pivot(index='Ano', columns='k', values='Razão M/H calculável')
t7.columns = [f'k = {k}' for k in KS]
print('\nTabela 7 — UFs com razão mulheres/homens calculável (entre as UFs com os dois sexos presentes):')
display(t7)

# Tabela 8 — supressão complementar
t8 = sim_uf[['Ano', 'k', 'comp_cel_H', 'comp_cel_M', 'comp_oculto_H', 'comp_oculto_M', 'comp_part_visivel']].copy()
t8.columns = ['Ano', 'k', 'Células ocultas (H)', 'Células ocultas (M)', '% vínculos ocultos H', '% vínculos ocultos M', 'Participação feminina visível (%)']
print('\nTabela 8 — supressão complementar (heurística; inclui células zero ocultadas por dedução):')
display(t8.round(2).style.hide(axis='index'))
TABELAS['T6_vies_participacao'] = t6.reset_index()
TABELAS['T7_razao_calculavel'] = t7.reset_index()
TABELAS['T8_complementar'] = t8

## 4.4 Agregação e cruzamentos mais finos
Compara quatro níveis de detalhe, com a mesma regra de supressão primária: **sexo × UF**, **sexo × região**, **sexo × região × escolaridade (grupos)** e **sexo × região × escolaridade (códigos detalhados)**.

In [ ]:
# 4.4.1 Consulta: região × sexo × escolaridade
sql_cel = """
SELECT regiao_geografica AS regiao, "Sexo - Código" AS sexo_cod,
       "Escolaridade Após 2005 - Código" AS esc_cod, COUNT(*) AS n
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2')
  AND regiao_geografica NOT IN ('NI','')
GROUP BY regiao_geografica, "Sexo - Código", "Escolaridade Após 2005 - Código"
"""
df_cel = query_anos(sql_cel)
df_cel['sexo'] = decodificar(df_cel['sexo_cod'], SEXO)
df_cel['esc_cod'] = df_cel['esc_cod'].astype(str)
df_cel['grupo_esc'] = decodificar(df_cel['esc_cod'], ESCOLARIDADE_GRUPO).fillna('Não informado')
assert set(df_cel['sexo'].unique()) <= set(SEXOS), f"Rótulos de sexo inesperados: {df_cel['sexo'].unique()}"
nao_reg = set(df_cel['regiao'].unique()) - set(REGIOES)
assert not nao_reg, f'Regiões inesperadas: {nao_reg}'

# Conferência: região via banco × soma das UFs
reg_banco = df_cel.groupby('ano')['n'].sum()
print('Vínculos com região válida:', {a: br(v) for a, v in reg_banco.items()},
      '| Total da ocupação:', {r.Ano: br(r.Total) for r in df_cbo.itertuples()})

In [ ]:
# 4.4.2 Quatro níveis de detalhe
universos = {'regiao': REGIOES,
             'grupo_esc': sorted(df_cel['grupo_esc'].unique()),
             'esc_cod': sorted(df_cel['esc_cod'].unique())}
niveis = {
    'Sexo × UF':                                  (grade_a, ['UF']),
    'Sexo × Região':                              (grade_completa(df_cel, ['regiao'], universos), ['regiao']),
    'Sexo × Região × Escolaridade (grupos)':      (grade_completa(df_cel, ['regiao', 'grupo_esc'], universos), ['regiao', 'grupo_esc']),
    'Sexo × Região × Escolaridade (códigos)':     (grade_completa(df_cel, ['regiao', 'esc_cod'], universos), ['regiao', 'esc_cod']),
}
res_niveis = pd.concat([supressao_primaria(g).assign(Nível=nome) for nome, (g, _) in niveis.items()], ignore_index=True)
res_niveis['Sexo'] = res_niveis['Sexo'].map(ROTULO_SEXO)

resumo_niv = (res_niveis[res_niveis['Ano'] == 2025]
              .pivot_table(index=['Nível', 'k'], columns='Sexo',
                           values=['celulas', 'celulas_suprimidas', '% celulas suprimidas', '% vinculos ocultados'])
              .round(1))
print('Supressão primária por nível de detalhe — 2025:')
display(resumo_niv)
TABELAS['T9_niveis_de_detalhe'] = res_niveis

In [ ]:
# 4.4.3 Figura: % de vínculos ocultados por nível de detalhe (2025)
ordem_niv = list(niveis.keys())
rotulos_curtos = ['Sexo×UF', 'Sexo×Região', 'Sexo×Região×\nEsc. (grupos)', 'Sexo×Região×\nEsc. (códigos)']
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, k in zip(axes, KS):
    d = res_niveis[(res_niveis['Ano'] == 2025) & (res_niveis['k'] == k)]
    pv = d.pivot(index='Nível', columns='Sexo', values='% vinculos ocultados').reindex(ordem_niv)[['Homens', 'Mulheres']]
    pv.plot(kind='bar', ax=ax, width=0.7, color=[CORES_GENERO['Masculino'], CORES_GENERO['Feminino']])
    ax.set_title(f'k = {k}'); ax.set_xlabel(''); ax.set_xticklabels(rotulos_curtos, rotation=0, fontsize=8)
    ax.legend(title='Sexo')
axes[0].set_ylabel('% de vínculos ocultados (2025)')
plt.suptitle('Impacto da supressão primária por nível de detalhe — CBO 2112-20', fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout(); salvar_fig('fig2_supressao_por_nivel.png'); plt.show()

In [ ]:
# 4.4.4 Agregação UF → região: simulação completa (primária + complementar) no nível região
df_reg_sexo = (df_cel.groupby(['ano', 'regiao', 'sexo'])['n'].sum().unstack('sexo').fillna(0).astype(int)
               .rename(columns={'Masculino': 'Homens', 'Feminino': 'Mulheres'}).reset_index().rename(columns={'ano': 'Ano'}))
print('Vínculos por região, sexo e ano:')
display(df_reg_sexo.assign(**{'%M': lambda x: (100 * x['Mulheres'] / (x['Homens'] + x['Mulheres'])).round(1)}))

sim_reg = rodar_simulacao(df_reg_sexo, 'regiao', REGIOES)
comp_agreg = pd.DataFrame({
    'Ano': sim_uf['Ano'], 'k': sim_uf['k'],
    'UF: razão M/H calculável': sim_uf['prim_razao_calculavel'].astype(str) + '/' + sim_uf['prim_razao_base'].astype(str),
    'Região: razão M/H calculável': sim_reg['prim_razao_calculavel'].astype(str) + '/' + sim_reg['prim_razao_base'].astype(str),
    'UF: células M suprimidas': sim_uf['prim_cel_M'], 'Região: células M suprimidas': sim_reg['prim_cel_M'],
})
print('\nAgregar UF → região: comparabilidade preservada?')
display(comp_agreg.style.hide(axis='index'))
TABELAS['T10_agregacao_uf_regiao'] = comp_agreg

# 5. Análises complementares (apêndice)
Descritivas, sobre a ocupação. Percentuais **dentro de cada sexo**; leia com cautela, pois as categorias têm poucos vínculos.

## 5.1 Participação feminina por região

In [ ]:
reg_ano = df_reg_sexo.assign(Total=lambda x: x['Homens'] + x['Mulheres'])
reg_ano['pct_fem'] = 100 * reg_ano['Mulheres'] / reg_ano['Total']
pivot_reg = reg_ano.pivot(index='regiao', columns='Ano', values='pct_fem').reindex(REGIOES)
print('Participação feminina (%) por região:'); display(pivot_reg.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot_reg, annot=True, fmt='.1f', cmap='RdYlGn', linewidths=0.5, ax=ax,
            cbar_kws={'label': '% Feminino'}, vmin=15, vmax=40)
ax.set_title('Participação Feminina (%) por Região\nCientista de Dados (CBO 2112-20)')
plt.tight_layout(); salvar_fig('fig3_participacao_regiao.png'); plt.show()

## 5.2 Remuneração por sexo — comparação com o Relatório de Transparência Salarial
**Referência (MTE/Ministério das Mulheres, painel de dez/2025, Brasil):** razão M/H da remuneração média = **78,7%** (gap de **21,3%**); razão do salário contratual mediano = 85,7% (gap de 14,3%, calculado a partir da tabela do painel: R$ 2.105,20 ÷ R$ 2.455,70). Universo: estabelecimentos com **100 ou mais empregados**, excluídos intermitentes e avulsos; remuneração média com valores entre 0,3 e 150 salários mínimos; razão só calculada com **pelo menos 3 mulheres e 3 homens**.

**Como o notebook replica o painel:** (i) restringe a estabelecimentos com 100+ empregados (códigos de porte 7 a 10), que é o universo comparável; (ii) aplica o corte de 0,3 a 150 salários mínimos do ano; (iii) aplica a regra de 3 mulheres e 3 homens; (iv) compara **média com média** (principal) e mediana com mediana (complementar). Também mostra o resultado para todos os estabelecimentos, como contexto.

> **Limitações:** `Vl Rem Média Nom` é a remuneração média **nominal** do vínculo, não a de dezembro a valores constantes usada no painel, e o painel usa salário contratual mensal para a mediana. Por isso, a comparação com o Brasil é **indicativa**, e os níveis em reais não são comparáveis entre anos sem deflacionar. A razão M/H dentro de cada ano é válida.

In [ ]:
SM = {2023: 1320.0, 2024: 1412.0, 2025: 1518.0}      # salário mínimo vigente no ano-base
GAP_MEDIA_BRASIL_PCT   = 21.3                          # painel dez/2025: razão M/H remuneração média = 78,7%
GAP_MEDIANA_BRASIL_PCT = 14.3                          # painel dez/2025: razão M/H salário mediano = 85,7%
PORTES_100_MAIS = ('7', '8', '9', '10')                # 100-249, 250-499, 500-999, 1000+

sql_sal = """
SELECT "Sexo - Código" AS sexo_cod, COUNT(*) AS n,
       AVG(rem) AS media_sal, MEDIAN(rem) AS mediana_sal
FROM (SELECT "Sexo - Código", "Tamanho Estabelecimento - Código" AS tam,
             TRY_CAST(REPLACE("Vl Rem Média Nom", ',', '.') AS DOUBLE) AS rem
      FROM rais_vinc_brasil
      WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2'))
WHERE rem >= {SM_MIN} * 0.3 AND rem < {SM_MIN} * 150 {FILTRO_PORTE}
GROUP BY "Sexo - Código"
"""
linhas = []
for ano in ANOS:
    for universo, filtro in [('100+ empregados (comparável ao painel)',
                              "AND tam IN ('" + "','".join(PORTES_100_MAIS) + "')"),
                             ('Todos os estabelecimentos', '')]:
        sql = sql_sal.replace('{SM_MIN}', str(SM[ano])).replace('{FILTRO_PORTE}', filtro)
        d = query_anos(sql, anos=[ano])
        d['sexo'] = decodificar(d['sexo_cod'], SEXO)
        d = d.set_index('sexo').reindex(SEXOS)
        nH, nM = d.loc['Masculino', 'n'], d.loc['Feminino', 'n']
        nH = 0 if pd.isna(nH) else int(nH); nM = 0 if pd.isna(nM) else int(nM)
        calc = nH >= 3 and nM >= 3                     # regra do painel: ≥3 mulheres e ≥3 homens
        linha = {'Ano': ano, 'Universo': universo, 'n Homens': nH, 'n Mulheres': nM, 'Razão calculável': 'sim' if calc else 'não (<3)'}
        if calc:
            linha.update({
                'Média H (R$)': d.loc['Masculino', 'media_sal'], 'Média M (R$)': d.loc['Feminino', 'media_sal'],
                'Razão M/H média (%)': 100 * d.loc['Feminino', 'media_sal'] / d.loc['Masculino', 'media_sal'],
                'Mediana H (R$)': d.loc['Masculino', 'mediana_sal'], 'Mediana M (R$)': d.loc['Feminino', 'mediana_sal'],
                'Razão M/H mediana (%)': 100 * d.loc['Feminino', 'mediana_sal'] / d.loc['Masculino', 'mediana_sal']})
        linhas.append(linha)
df_gap = pd.DataFrame(linhas)
df_gap['Gap média (%)']    = 100 - df_gap.get('Razão M/H média (%)')
df_gap['Gap mediana (%)']  = 100 - df_gap.get('Razão M/H mediana (%)')
df_gap['Gap média − Brasil (p.p.)'] = df_gap['Gap média (%)'] - GAP_MEDIA_BRASIL_PCT
print(f'Referência Brasil (100+): gap médio {GAP_MEDIA_BRASIL_PCT}% | gap mediano {GAP_MEDIANA_BRASIL_PCT}%')
display(df_gap.round(1))
TABELAS['A1_remuneracao'] = df_gap

# Figura: razão M/H (média e mediana) na ocupação, universo comparável vs Brasil
comp = df_gap[df_gap['Universo'].str.startswith('100+')].dropna(subset=['Razão M/H média (%)'])
fig, ax = plt.subplots(figsize=(8, 5))
if len(comp):
    ax.plot(comp['Ano'], comp['Razão M/H média (%)'], '-o', color='#E91E63', lw=2.5, label='Cientista de Dados — média')
    ax.plot(comp['Ano'], comp['Razão M/H mediana (%)'], '--o', color='#8E24AA', lw=2, label='Cientista de Dados — mediana')
ax.axhline(100 - GAP_MEDIA_BRASIL_PCT, color='black', ls='--', lw=1.2, label='Brasil, média (78,7%)')
ax.axhline(100 - GAP_MEDIANA_BRASIL_PCT, color='gray', ls=':', lw=1.2, label='Brasil, mediana (85,7%)')
ax.axhline(100, color='black', lw=0.6, alpha=0.4)
ax.set_title('Razão salarial M/H (%) — estabelecimentos com 100+ empregados'); ax.set_xlabel('Ano'); ax.set_ylabel('Mulheres ÷ Homens (%)')
ax.set_xticks(ANOS); ax.legend(fontsize=8)
plt.tight_layout(); salvar_fig('fig4_remuneracao.png'); plt.show()

## 5.3 Raça/cor por sexo
> Códigos de raça/cor na RAIS **não são sequenciais** (1=Indígena, 2=Branca, 4=Preta, 6=Amarela, 8=Parda, 9=Não identificado); 3, 5 e 7 não existem no layout oficial. Excluídos os códigos 9 e −1.

In [ ]:
sql_raca = """
SELECT "Sexo - Código" AS sexo_cod, "Raça Cor - Código" AS raca_cod, COUNT(*) AS vinculos
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2')
  AND "Raça Cor - Código" NOT IN ('-1','9','') AND "Raça Cor - Código" IS NOT NULL
GROUP BY "Sexo - Código", "Raça Cor - Código"
"""
df_raca = query_anos(sql_raca)
df_raca['sexo']  = decodificar(df_raca['sexo_cod'], SEXO)
df_raca['grupo'] = decodificar(df_raca['raca_cod'], RACA_COR_GRUPO)

df_grp = df_raca.groupby(['ano', 'sexo', 'grupo'])['vinculos'].sum().reset_index()
df_grp['pct'] = (df_grp['vinculos'] / df_grp.groupby(['ano', 'sexo'])['vinculos'].transform('sum') * 100).round(2)
display(df_grp[df_grp['ano'] == 2025].pivot(index='grupo', columns='sexo', values='pct').round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ano in zip(axes, [2024, 2025]):
    pv = df_grp[df_grp['ano'] == ano].pivot(index='grupo', columns='sexo', values='pct').fillna(0)
    pv.plot(kind='bar', ax=ax, color=[CORES_GENERO['Feminino'], CORES_GENERO['Masculino']], width=0.6)
    ax.set_title(f'Raça/Cor por Sexo — {ano}\nCientista de Dados (CBO 2112-20)')
    ax.set_xlabel('Grupo'); ax.set_ylabel('% dentro do sexo'); ax.legend(title='Sexo')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')
plt.tight_layout(); salvar_fig('fig5_raca_sexo.png'); plt.show()

## 5.4 Escolaridade por sexo

In [ ]:
sql_esc = """
SELECT "Sexo - Código" AS sexo_cod, "Escolaridade Após 2005 - Código" AS esc_cod, COUNT(*) AS vinculos
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2')
  AND "Escolaridade Após 2005 - Código" IS NOT NULL AND "Escolaridade Após 2005 - Código" != '-1'
GROUP BY "Sexo - Código", "Escolaridade Após 2005 - Código"
"""
df_esc = query_anos(sql_esc)
df_esc['sexo'] = decodificar(df_esc['sexo_cod'], SEXO)
df_esc['grupo_esc'] = decodificar(df_esc['esc_cod'], ESCOLARIDADE_GRUPO)

GRUPO_ESC_ORDEM = ['Até fundamental', 'Médio', 'Superior', 'Pós-graduação']
df_eg = df_esc.groupby(['ano', 'sexo', 'grupo_esc'])['vinculos'].sum().reset_index()
df_eg['pct'] = (df_eg['vinculos'] / df_eg.groupby(['ano', 'sexo'])['vinculos'].transform('sum') * 100).round(1)
display(df_eg[df_eg['ano'] == 2025].pivot(index='grupo_esc', columns='sexo', values='pct').reindex(GRUPO_ESC_ORDEM).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ano in zip(axes, [2024, 2025]):
    pv = df_eg[df_eg['ano'] == ano].pivot(index='grupo_esc', columns='sexo', values='pct').fillna(0).reindex(GRUPO_ESC_ORDEM)
    pv.plot(kind='bar', ax=ax, color=[CORES_GENERO['Feminino'], CORES_GENERO['Masculino']], width=0.6)
    ax.set_title(f'Escolaridade por Sexo — {ano}\nCientista de Dados (CBO 2112-20)')
    ax.set_xlabel(''); ax.set_ylabel('% dentro do sexo'); ax.legend(title='Sexo')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')
plt.tight_layout(); salvar_fig('fig6_escolaridade_sexo.png'); plt.show()

## 5.5 Porte do estabelecimento por sexo
> Proxy de senioridade e formalização. Código `1` = zero empregados; código `2` = até 4 (layout oficial MTE).

In [ ]:
sql_tam = """
SELECT "Sexo - Código" AS sexo_cod, "Tamanho Estabelecimento - Código" AS tam_cod, COUNT(*) AS vinculos
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2')
  AND "Tamanho Estabelecimento - Código" IS NOT NULL AND "Tamanho Estabelecimento - Código" != '-1'
GROUP BY "Sexo - Código", "Tamanho Estabelecimento - Código"
"""
df_tam = query_anos(sql_tam)
df_tam['sexo']  = decodificar(df_tam['sexo_cod'], SEXO)
df_tam['porte'] = decodificar(df_tam['tam_cod'], TAMANHO_ESTAB_PORTE)
df_tp = df_tam.groupby(['ano', 'sexo', 'porte'])['vinculos'].sum().reset_index()
df_tp['pct'] = (df_tp['vinculos'] / df_tp.groupby(['ano', 'sexo'])['vinculos'].transform('sum') * 100).round(1)
pv_t = (df_tp[df_tp['ano'] == 2025].pivot(index='porte', columns='sexo', values='pct').fillna(0).reindex(TAMANHO_PORTE_ORDEM))
display(pv_t.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
pv_t.plot(kind='bar', ax=ax, color=[CORES_GENERO['Feminino'], CORES_GENERO['Masculino']], width=0.6)
ax.set_title('Porte do Estabelecimento por Sexo — 2025\nCientista de Dados (CBO 2112-20)')
ax.set_xlabel('Porte'); ax.set_ylabel('% dentro do sexo'); ax.legend(title='Sexo'); plt.xticks(rotation=0)
plt.tight_layout(); salvar_fig('fig7_porte_estabelecimento.png'); plt.show()

## 5.6 Tipo de vínculo por sexo
Distribuição das modalidades de contratação (CLT, estatutário, temporário etc.) em 2025, como proxy de estabilidade e modelo de contratação.

In [ ]:
sql_vinc = """
SELECT "Sexo - Código" AS sexo_cod, "Tipo Vínculo - Código" AS vinc_cod, COUNT(*) AS vinculos
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE} AND "Sexo - Código" IN ('1','2')
  AND "Tipo Vínculo - Código" IS NOT NULL AND "Tipo Vínculo - Código" != '-1'
GROUP BY "Sexo - Código", "Tipo Vínculo - Código"
"""
df_vinc = query_anos(sql_vinc)
df_vinc['sexo'] = decodificar(df_vinc['sexo_cod'], SEXO)
df_vinc['grupo_vinculo'] = decodificar(df_vinc['vinc_cod'], TIPO_VINCULO_GRUPO)
df_vg = df_vinc.groupby(['ano', 'sexo', 'grupo_vinculo'])['vinculos'].sum().reset_index()
df_vg['pct'] = (df_vg['vinculos'] / df_vg.groupby(['ano', 'sexo'])['vinculos'].transform('sum') * 100).round(1)
pv_vg = df_vg[df_vg['ano'] == 2025].pivot(index='grupo_vinculo', columns='sexo', values='pct').fillna(0)
display(pv_vg.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
pv_vg.plot(kind='bar', ax=ax, color=[CORES_GENERO['Feminino'], CORES_GENERO['Masculino']], width=0.6)
ax.set_title('Tipo de Vínculo por Sexo — 2025\nCientista de Dados (CBO 2112-20)')
ax.set_xlabel('Tipo de Vínculo'); ax.set_ylabel('% dentro do sexo'); ax.legend(title='Sexo'); plt.xticks(rotation=20, ha='right')
plt.tight_layout(); salvar_fig('fig8_tipo_vinculo.png'); plt.show()

## 5.7 Completude das variáveis sensíveis
Ausências em dados têm consequências distributivas (D'Ignazio & Klein, 2020). Raça/cor: o código 9 (não identificado) é distinto de −1 (ignorado); ambos comprometem análises de equidade racial.

In [ ]:
sql_compl = """
SELECT COUNT(*) AS total,
       SUM(CASE WHEN "Sexo - Código" NOT IN ('1','2') OR "Sexo - Código" IS NULL THEN 1 ELSE 0 END) AS sem_sexo,
       SUM(CASE WHEN TRY_CAST(REPLACE("Vl Rem Média Nom",',','.') AS DOUBLE) IS NULL
                  OR TRY_CAST(REPLACE("Vl Rem Média Nom",',','.') AS DOUBLE) <= 0 THEN 1 ELSE 0 END) AS sem_salario,
       SUM(CASE WHEN "Raça Cor - Código" IS NULL OR "Raça Cor - Código" IN ('-1','9','') THEN 1 ELSE 0 END) AS sem_raca,
       SUM(CASE WHEN "Escolaridade Após 2005 - Código" IS NULL OR "Escolaridade Após 2005 - Código" = '-1' THEN 1 ELSE 0 END) AS sem_escolaridade,
       SUM(CASE WHEN "Município - Código" IS NULL OR "Município - Código" = '' THEN 1 ELSE 0 END) AS sem_municipio
FROM rais_vinc_brasil
WHERE "CBO 2002 Ocupação - Código" = '{CBO}' AND {ESTOQUE}
"""
df_compl = query_anos(sql_compl)
for c in ['sem_sexo', 'sem_salario', 'sem_raca', 'sem_escolaridade', 'sem_municipio']:
    df_compl[f'pct_{c}'] = (df_compl[c] / df_compl['total'] * 100).round(2)
cols = ['ano', 'total', 'pct_sem_sexo', 'pct_sem_salario', 'pct_sem_raca', 'pct_sem_escolaridade', 'pct_sem_municipio']
display(df_compl[cols])

df_plot = df_compl[cols[2:]].copy(); df_plot.index = df_compl['ano']
df_plot.columns = ['Sexo', 'Salário', 'Raça/Cor', 'Escolaridade', 'Município']
fig, ax = plt.subplots(figsize=(10, 5))
df_plot.T.plot(kind='bar', ax=ax, width=0.7, colormap='Set2')
ax.set_title('Taxa de Incompletude (%) por Variável e Ano\nCientista de Dados (CBO 2112-20)')
ax.set_xlabel('Variável'); ax.set_ylabel('% de registros sem informação'); ax.legend(title='Ano'); plt.xticks(rotation=20, ha='right')
plt.tight_layout(); salvar_fig('fig9_completude_variaveis.png'); plt.show()

# 6. Resumo executivo e exportação

In [ ]:
print('=' * 64)
print('RESUMO EXECUTIVO — CBO 2112-20 Cientista de Dados')
print('Fonte: RAIS Microdados 2023–2025 (MTE/eSocial)')
print('=' * 64)
for _, r in df_cbo.iterrows():
    cresc = '' if pd.isna(r['Crescimento (%)']) else f"  | Crescimento: +{r['Crescimento (%)']:.1f}%"
    print(f"{int(r['Ano'])} | {br(r['Total'])} vínculos | Fem. {r['% Feminino']:.1f}% | Masc. {r['% Masculino']:.1f}%{cresc}")

print('\nRazão salarial M/H (100+ empregados; Brasil: média 78,7% | mediana 85,7%):')
for _, r in df_gap[df_gap['Universo'].str.startswith('100+')].iterrows():
    if r['Razão calculável'] == 'sim':
        print(f"  {int(r['Ano'])}: média {r['Razão M/H média (%)']:.1f}% | mediana {r['Razão M/H mediana (%)']:.1f}% (n H={r['n Homens']}, n M={r['n Mulheres']})")
    else:
        print(f"  {int(r['Ano'])}: razão não calculável (n H={r['n Homens']}, n M={r['n Mulheres']}; regra ≥3 de cada sexo)")

r25 = df_compl[df_compl['ano'] == 2025].iloc[0]
print('\nCompletude (2025): ' + ' | '.join(f"{n} {r25['pct_' + c]:.2f}%" for n, c in
      [('sem sexo', 'sem_sexo'), ('sem salário', 'sem_salario'), ('sem raça/cor', 'sem_raca'), ('sem escolaridade', 'sem_escolaridade')]))

print('\nSupressão primária em 2025 (% de vínculos ocultados, mulheres / homens):')
for nome in niveis:
    for k in KS:
        d = res_niveis[(res_niveis['Nível'] == nome) & (res_niveis['Ano'] == 2025) & (res_niveis['k'] == k)].set_index('Sexo')
        print(f"  {nome:<42} k={k:<2}: {d.loc['Mulheres', '% vinculos ocultados']:5.1f}% / {d.loc['Homens', '% vinculos ocultados']:5.1f}%"
              f"  | células M suprimidas: {int(d.loc['Mulheres', 'celulas_suprimidas'])}/{int(d.loc['Mulheres', 'celulas'])}")
print('=' * 64)

In [ ]:
# Exportação das tabelas finais para um único Excel (para o artigo)
caminho_xlsx = os.path.join(TAB_DIR, 'tabelas_tcc_rais.xlsx')
with pd.ExcelWriter(caminho_xlsx, engine='openpyxl') as xw:
    for nome, t in TABELAS.items():
        t.to_excel(xw, sheet_name=nome[:31], index=False)
print(f'💾 {caminho_xlsx}  ({len(TABELAS)} abas)')